# HỆ THỐNG DỰ ĐOÁN XÉT TUYỂN ĐẠI HỌC VIỆT NAM
# PHÂN TÍCH DỮ LIỆU GOOGLE TRENDS: TÍN HIỆU SỰ QUAN TÂM CỦA XÃ HỘI

---

## 🎯 Mục tiêu của Notebook này
Notebook này thực hiện quy trình **Khám phá Dữ liệu (Exploratory Data Analysis - EDA)** và **Khai phá Đặc trưng (Feature Engineering)** từ nguồn dữ liệu Google Trends đã được thu thập và chuẩn hóa trong **Data Lake** và **Data Warehouse DuckDB**.

Dữ liệu này đóng vai trò đại diện cho biến số: **"Sự quan tâm của xã hội đối với ngành/trường"**, phục vụ cho bài toán dự đoán khả năng trúng tuyển Đại học:
* **Giả thuyết cốt lõi**: Ngành học hoặc Trường đại học nào càng nhận được sự quan tâm lớn của xã hội thì mức độ cạnh tranh càng khốc liệt $\rightarrow$ Điểm chuẩn trúng tuyển có xu hướng bị đẩy lên cao.
* **Các chiều dữ liệu phân tích**:
  1. **Chuỗi thời gian (Time Series)**: Biến động xu hướng qua 5 năm (262 tuần) và tính mùa vụ tuyển sinh (tháng 5 – tháng 8).
  2. **Địa lý (Spatial/Geographical)**: Phân bố mức độ quan tâm trên 63 tỉnh/thành phố tại Việt Nam.
  3. **Tương quan liên ngành (Correlation)**: Mối quan hệ giữa sự quan tâm ngành học, trường đại học và tín hiệu thị trường việc làm.

---

## 🟢 BƯỚC 1: Thiết lập Môi trường & Khai báo Thư viện

### 💡 Giải thích:
Trong bước này, chúng ta nạp các thư viện phân tích cốt lõi:
* `duckdb`: Đọc và truy vấn nhanh cơ sở dữ liệu Data Warehouse không cần bật server.
* `pandas` & `numpy`: Xử lý bảng dữ liệu và tính toán thống kê ma trận.
* `matplotlib` & `seaborn`: Trực quan hóa dữ liệu với biểu đồ chuỗi thời gian, biểu đồ cột và bản đồ nhiệt (heatmap).
* Đồng thời cấu hình font chữ hiển thị tốt tiếng Việt trên hệ điều hành Windows.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập font và giao diện trực quan hóa đồ thị chuẩn tiếng Việt
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Đường dẫn tới Data Warehouse DuckDB
DUCKDB_PATH = '../data/warehouse/trends.duckdb' if os.path.exists('../data/warehouse/trends.duckdb') else 'data/warehouse/trends.duckdb'
print(f'Môi trường phân tích sẵn sàng. Kết nối DuckDB tại: {DUCKDB_PATH}')

## 🟢 BƯỚC 2: Kết nối Data Warehouse & Khám phá Cấu trúc Dữ liệu

### 💡 Giải thích:
Chúng ta kết nối ở chế độ `read_only=True` để tránh xung đột khóa file (file lock). Sau đó truy vấn danh sách tất cả các bảng dữ liệu đã được thu thập và chuẩn hóa thang đo toàn cục bằng thuật toán **Anchor Normalization**.

In [ ]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# Liệt kê các bảng hiện có trong Data Warehouse
tables_df = con.execute('SHOW TABLES;').fetchdf()
print('=== DANH SÁCH CÁC BẢNG DỮ LIỆU TRONG DUCKDB ===')
display(tables_df)

# Xem lịch sử ghi nhận metadata và số lượng dòng của từng danh mục
sync_log = con.execute('SELECT * FROM sys_trends_sync_log ORDER BY sync_time DESC').fetchdf()
print('\n=== NHẬT KÝ ĐỒNG BỘ DỮ LIỆU (AUDIT LOG) ===')
display(sync_log.head(5))

## 🟢 BƯỚC 3: Phân tích Xu hướng Chuỗi Thời gian các Ngành Đào tạo (Time Series Analysis)

### 💡 Giải thích:
Chúng ta đọc bảng `trends_time_majors_it` chứa dữ liệu 262 tuần (5 năm từ 2021 đến 2026).
* **Ý nghĩa**: Quan sát mức độ quan tâm của xã hội biến động như thế nào qua các năm.
* **Các ngành so sánh**: `Công nghệ thông tin` (ngành mốc), `Khoa học máy tính`, `Trí tuệ nhân tạo`, `Khoa học dữ liệu`, `An toàn thông tin`.

In [ ]:
# Đọc chuỗi thời gian các ngành học
df_majors = con.execute('SELECT * FROM trends_time_majors_it ORDER BY date ASC').fetchdf()
df_majors['date'] = pd.to_datetime(df_majors['date'])
df_majors = df_majors.set_index('date')

print(f'Kích thước dữ liệu ngành học: {df_majors.shape[0]} tuần x {df_majors.shape[1]} cột')
display(df_majors.tail(5))

# Vẽ đồ thị so sánh xu hướng tìm kiếm 5 năm qua
plt.figure(figsize=(15, 6))
selected_majors = ['Công nghệ thông tin', 'Khoa học máy tính', 'Trí tuệ nhân tạo', 'Khoa học dữ liệu', 'An toàn thông tin']
for col in selected_majors:
    if col in df_majors.columns:
        plt.plot(df_majors.index, df_majors[col], label=col, linewidth=2)

plt.title('Xu hướng Mức độ Quan tâm của Xã hội đối với các Ngành CNTT (2021 - 2026)', fontsize=14, fontweight='bold')
plt.xlabel('Thời gian', fontsize=12)
plt.ylabel('Chỉ số Quan tâm Chuẩn hóa [0 - 100]', fontsize=12)
plt.legend(loc='upper left', frameon=True, fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 📊 Nhận xét kết quả Bước 3:
1. **Ngành mốc `Công nghệ thông tin`** luôn duy trì lượng quan tâm áp đảo so với các chuyên ngành hẹp.
2. **Ngành `Trí tuệ nhân tạo` và `Khoa học dữ liệu`** có xu hướng tăng trưởng rõ rệt qua từng năm, phản ánh làn sóng công nghệ mới bắt đầu thu hút sự quan tâm của phụ huynh và thí sinh.
3. Đồ thị xuất hiện **các đỉnh nhọn lặp lại có chu kỳ** vào cùng một thời điểm mỗi năm.

## 🟢 BƯỚC 4: Nhận diện Tính Mùa vụ Tuyển sinh (Seasonality & Admission Spikes)

### 💡 Giải thích:
Kỳ thi tốt nghiệp THPT và xét tuyển Đại học tại Việt Nam diễn ra cố định vào mùa hè hàng năm:
* **Tháng 5 - 6**: Giai đoạn tìm hiểu thông tin xét tuyển sớm, thi tốt nghiệp THPT.
* **Tháng 7**: Bộ GD&ĐT công bố phổ điểm $\rightarrow$ **Đỉnh sóng tìm hiểu điểm chuẩn và lọc ảo nguyện vọng**.
* **Tháng 8**: Các trường công bố điểm chuẩn chính thức.

Chúng ta sẽ gom nhóm dữ liệu theo tháng trong năm (từ Tháng 1 đến Tháng 12) để chứng minh tính mùa vụ này.

In [ ]:
# Trích xuất tháng trong năm và tính giá trị trung bình theo tháng
df_season = df_majors.copy()
df_season['month'] = df_season.index.month

monthly_pattern = df_season.groupby('month')[selected_majors].mean()

plt.figure(figsize=(12, 5))
for col in selected_majors:
    if col in monthly_pattern.columns:
        plt.plot(monthly_pattern.index, monthly_pattern[col], marker='o', label=col, linewidth=2.2)

plt.title('Tính Mùa vụ: Mức độ Quan tâm Trung bình theo từng Tháng trong Năm', fontsize=13, fontweight='bold')
plt.xlabel('Tháng trong năm (1 - 12)', fontsize=11)
plt.ylabel('Chỉ số Tìm kiếm Trung bình', fontsize=11)
plt.xticks(range(1, 13), [f'Tháng {i}' for i in range(1, 13)])
plt.axvspan(6.5, 8.5, color='yellow', alpha=0.2, label='Mùa cao điểm Tuyển sinh (Tháng 7 - 8)')
plt.legend(loc='upper right', frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 📊 Nhận xét kết quả Bước 4:
* Biểu đồ chứng minh rõ ràng: **Tháng 7 và Tháng 8** luôn là giai đoạn có chỉ số quan tâm bùng nổ vượt trội so với các tháng còn lại trong năm.
* Đây chính là bằng chứng xác thực cho **Giả thuyết nghiên cứu**: Lượng tìm kiếm trên Google Trends phản ánh chính xác hành vi nộp hồ sơ và cạnh tranh nguyện vọng của thí sinh.

## 🟢 BƯỚC 5: So sánh Sức hút giữa các Trường Đại học Trọng điểm

### 💡 Giải thích:
Chúng ta đọc bảng `trends_time_universities_it` để so sánh mức độ quan tâm của xã hội đối với các trường Đại học đào tạo CNTT hàng đầu:
* `Đại học Bách Khoa` (Bách Khoa Hà Nội & Bách Khoa TP.HCM)
* `Đại học Công nghệ Thông tin` (UIT)
* `Đại học FPT`
* `Đại học Khoa học Tự nhiên`
* `Học viện Bưu chính Viễn thông`

In [ ]:
# Đọc chuỗi thời gian các trường đại học
df_unis = con.execute('SELECT * FROM trends_time_universities_it ORDER BY date ASC').fetchdf()
df_unis['date'] = pd.to_datetime(df_unis['date'])
df_unis = df_unis.set_index('date')

plt.figure(figsize=(15, 6))
top_unis = ['Đại học Bách Khoa', 'Đại học Công nghệ Thông tin', 'Đại học FPT', 'Đại học Khoa học Tự nhiên', 'Học viện Bưu chính Viễn thông']
for col in top_unis:
    if col in df_unis.columns:
        plt.plot(df_unis.index, df_unis[col], label=col, linewidth=2)

plt.title('So sánh Mức độ Quan tâm giữa các Trường Đại học Đào tạo CNTT (2021 - 2026)', fontsize=14, fontweight='bold')
plt.xlabel('Thời gian', fontsize=12)
plt.ylabel('Chỉ số Quan tâm Chuẩn hóa [0 - 100]', fontsize=12)
plt.legend(loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 🟢 BƯỚC 6: Phân tích Không gian Địa lý (Phân bố 63 Tỉnh/Thành Việt Nam)

### 💡 Giải thích:
Đặc thù tuyển sinh tại Việt Nam có yếu tố vùng miền: thí sinh ở các tỉnh thành lớn hoặc các khu vực công nghệ phát triển thường có xu hướng tìm hiểu các ngành học mũi nhọn sớm hơn.
Chúng ta đọc bảng `trends_region_majors_it` để xác định **Top 10 Tỉnh/Thành phố** có chỉ số quan tâm cao nhất đến các ngành công nghệ cao (`Trí tuệ nhân tạo` và `Khoa học máy tính`).

In [ ]:
# Đọc dữ liệu phân bố địa lý theo tỉnh/thành
df_region = con.execute('SELECT * FROM trends_region_majors_it').fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Top 10 tỉnh/thành quan tâm nhất đến Trí tuệ nhân tạo
if 'Trí tuệ nhân tạo' in df_region.columns:
    top_ai = df_region.sort_values(by='Trí tuệ nhân tạo', ascending=False).head(10)
    sns.barplot(data=top_ai, x='geoName', y='Trí tuệ nhân tạo', ax=axes[0], palette='Blues_r')
    axes[0].set_title('Top 10 Tỉnh/Thành quan tâm đến "Trí tuệ nhân tạo"', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Tỉnh / Thành phố', fontsize=10)
    axes[0].set_ylabel('Chỉ số Quan tâm', fontsize=10)
    axes[0].tick_params(axis='x', rotation=35)

# 2. Top 10 tỉnh/thành quan tâm nhất đến Khoa học máy tính
if 'Khoa học máy tính' in df_region.columns:
    top_cs = df_region.sort_values(by='Khoa học máy tính', ascending=False).head(10)
    sns.barplot(data=top_cs, x='geoName', y='Khoa học máy tính', ax=axes[1], palette='Greens_r')
    axes[1].set_title('Top 10 Tỉnh/Thành quan tâm đến "Khoa học máy tính"', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Tỉnh / Thành phố', fontsize=10)
    axes[1].set_ylabel('Chỉ số Quan tâm', fontsize=10)
    axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

## 🟢 BƯỚC 7: Ma trận Tương quan & Khai phá Đặc trưng (Feature Engineering)

### 💡 Giải thích:
Để phục vụ huấn luyện mô hình Machine Learning (XGBoost, Suy luận mờ Takagi-Sugeno):
1. Chúng ta tính **Ma trận Tương quan Pearson** giữa các ngành đào tạo.
2. Phân tích xem sự gia tăng quan tâm của ngành này có kéo theo sự gia tăng của các ngành công nghệ liên quan hay không.
3. Trích xuất các đặc trưng chuỗi thời gian như: **Trung bình trượt 4 tuần (Rolling Mean)** và **Độ tăng trưởng tuần (Weekly Momentum)**.

In [ ]:
# 1. Ma trận tương quan giữa các ngành CNTT
numeric_cols = [c for c in df_majors.columns if c != 'isPartial']
corr_matrix = df_majors[numeric_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1, square=True, linewidths=0.5)
plt.title('Ma trận Tương quan Mức độ Quan tâm giữa các Ngành Đào tạo CNTT', fontsize=13, fontweight='bold')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

# 2. Trích xuất đặc trưng mới: Rolling Mean 4 tuần (Đại diện cho xu hướng 1 tháng vừa qua)
features_df = pd.DataFrame(index=df_majors.index)
features_df['it_raw'] = df_majors['Công nghệ thông tin']
features_df['it_rolling_4w'] = df_majors['Công nghệ thông tin'].rolling(window=4).mean()
features_df['it_momentum'] = df_majors['Công nghệ thông tin'].diff(periods=1)

print('=== ĐẶC TRƯNG MỚI ĐƯỢC TẠO RA CHO MÔ HÌNH MACHINE LEARNING ===')
display(features_df.dropna().tail(5))

## 🟢 BƯỚC 8: Tích hợp với Apache Spark (PySpark Pipeline)

### 💡 Giải thích:
Sau khi hoàn tất quá trình khám phá (EDA) trên DuckDB, dữ liệu Parquet chuẩn hóa trong **Data Lake Master Zone** (`data/lake/normalized/`) được chuyển tiếp trực tiếp vào luồng xử lý phân tán **Apache Spark (PySpark)** để ghép nối với dữ liệu điểm thi và điểm chuẩn các năm.

In [ ]:
print("""
MÃ NGUỒN MẪU NẠP DỮ LIỆU VÀO PYSPARK DATA ENGINE:
--------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg

# Khởi tạo Spark Session tối ưu bộ nhớ
spark = SparkSession.builder \
    .appName("AdmissionPrediction_TrendsPipeline") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Đọc bảng Master Parquet từ Data Lake Master Zone
parquet_path = "../data/lake/normalized/master_trends_majors_it.parquet"
df_spark = spark.read.parquet(parquet_path)

df_spark.printSchema()
df_spark.show(5)
--------------------------------------------------
""")

# Đóng kết nối DuckDB an toàn
con.close()
print('Quy trình phân tích Google Trends đã hoàn tất thành công!')